### Resumen - Beneficio por Empleado
> Para cada "Beneficio" se necesita saber la cantidad de empleados activos durante el año "2023" y "2024"

In [0]:
df_employees = spark.table("zenviro.silver.employees_py").alias("e")
display(df_employees)

In [0]:
df_benefits = spark.table("zenviro.silver.benefits_py").alias("b")
display(df_benefits)

In [0]:
df_employee_benefits = spark.table("zenviro.silver.employee_benefits_py").alias("eb")
display(df_employee_benefits)

In [0]:
from pyspark.sql.functions import col, year, count

df_benefits_by_employees_summary = (
    df_benefits
    .join(df_employee_benefits, col("b.benefit_id") == col("eb.benefit_id"), "inner")
    .join(df_employees, col("eb.employee_id") == col("e.employee_id"), "inner")
    .filter((year(col("eb.start_date")).isin(2023, 2024)) &
            (col("eb.end_date").isNull()))
    .groupBy(col("b.benefit_name"))
    .agg(
        count("e.employee_id").alias("total_employees")
    )
    .orderBy(col("total_employees").desc())
)
display(df_benefits_by_employees_summary)

In [0]:
df_benefits_by_employees_summary.writeTo("zenviro.gold.benefits_by_employees_summary_py").createOrReplace()

In [0]:
display(spark.table("zenviro.gold.benefits_by_employees_summary_py"))